# AsteroScale examples

A back-of-envelope calculator for asteroseismic scaling relations: give it any subset of a star's
observed parameters (with uncertainties), and it returns any other parameter(s) you ask for --
marginalizing over whatever wasn't given, using nested sampling (or, when everything given is exact,
skipping the sampler entirely for a fast point estimate).

See `README.md` in the package for the full API reference and caveats -- this notebook is a tour of
the main use cases.

In [ ]:
import asteroscale as ast
from asteroscale import relations

## 1. Basic usage: fully constrained star

Give mass, radius, Teff, and [Fe/H] with uncertainties; ask for numax and dnu. This is the simplest
case -- every fundamental parameter is directly constrained, so there's not much to marginalize
over.

In [ ]:
solver = ast.Solver(seed=0)

given = {"M": (1.0, 0.05), "R": (1.0, 0.02), "Teff": (5777, 50), "FeH": (0.0, 0.05)}
out = solver.solve(given, want=["numax", "dnu"])
ast.summarize(out)
fig = ast.plot_posterior(out)

## 2. Marginalizing over an unconstrained parameter

Leave radius out entirely. `solve()` marginalizes over it using its default prior (log-uniform) --
there's no separate "inverse" relation needed, the same forward model handles every direction.

In [ ]:
given = {"M": (1.0, 0.05), "Teff": (5777, 50), "FeH": (0.0, 0.05), "numax": (3090, 30)}
out = solver.solve(given, want=["R", "dnu", "rho"])
ast.summarize(out)
fig = ast.plot_posterior(out)

## 3. Combining seismology with Gaia astrometry and photometry

numax and dnu alone leave a mass-radius degeneracy (for a given Teff). Adding parallax and an
apparent Gaia G magnitude breaks it, since they constrain radius/luminosity independently through
the distance modulus.

In [ ]:
given = {
    "Teff": (5777, 50),
    "FeH": (0.0, 0.05),
    "numax": (3090, 30),
    "dnu": (135.1, 1.0),
    "plx": (10.0, 0.1),   # mas -> ~100 pc
    "G_mag": (9.90, 0.02),
}
out = solver.solve(given, want=["M", "R", "L"])
ast.summarize(out)
fig = ast.plot_posterior(out)

## 4. The top-level `solve()` shortcut

For one-off calculations, skip instantiating `Solver` yourself. This reuses a shared default solver
under the hood (unless you pass a `Solver` constructor argument such as `nlive`, `preset`,
`priors`, or `seed`, which creates a fresh solver).


In [ ]:
out = ast.solve(given, want=["L", "R"])
ast.summarize(out)

## 5. Gaia BP-RP color as a parallax-free constraint

Unlike `G_mag`, the BP-RP color is (mostly) distance-independent -- the distance modulus cancels
between the two bands -- so it constrains Teff/extinction without needing a parallax at all. Note it's
degenerate with extinction on its own (see the README caveats): a single color can't separate Teff
from reddening without an informative `A_G` prior.

In [ ]:
given = {
    "FeH": (0.0, 0.05),
    "numax": (3090, 30),
    "dnu": (135.1, 1.0),
    "BP_RP": (0.82, 0.02),
    "A_G": (0.05, 0.02),  # even a loose extinction estimate (e.g. from a dust map)
                            # breaks the reddening-Teff degeneracy and speeds this up a lot
}
out = ast.solve(given, want=["Teff", "M", "R"])
ast.summarize(out)
fig = ast.plot_posterior(out)

## 6. Metallicity corrections

`numax` and `dnu` both carry metallicity-dependent corrections (Viani et al. 2017 and
Guggenberger et al. 2016 respectively) -- small near solar composition, growing for metal-poor
stars.

In [ ]:
print("f_numax([Fe/H]=0.0): ", relations.f_numax(0.0))
print("f_numax([Fe/H]=-2.0):", relations.f_numax(-2.0))
print()
print("dnu_ref(Teff=4800, [Fe/H]=0.0): ", relations.dnu_ref(4800, 0.0))
print("dnu_ref(Teff=4800, [Fe/H]=-2.0):", relations.dnu_ref(4800, -2.0))

## 7. Flexible input types

Each value in `given` can be:

- **a plain number** -- treated as exactly known (see the fast point-estimate path below)
- **a `(mean, error)` tuple** -- wrapped as a Gaussian
- **any object with `.logpdf` and/or `.ppf`** -- used directly, e.g. a frozen
  `scipy.stats` distribution or one of the lightweight distributions supplied by AsteroScale.

Here a frozen SciPy normal distribution is mixed with ordinary Gaussian tuple constraints.
The same interface can represent asymmetric measurements using, for example, `scipy.stats.skewnorm`.

In [ ]:
import scipy.stats as st
given = {
    "Teff": st.norm(loc=5750, scale=40),
    "FeH": (0.0, 0.05),
    "numax": (3090, 30),
    "dnu": (135.1, 1.0),
}
out = solver.solve(given, want=["Teff", "M"])
ast.summarize(out)
fig = ast.plot_posterior(out)

## 8. Fast point estimates (no sampler)

If *every* given value is a plain scalar (no uncertainty), `solve()` skips Dynesty entirely: a direct
forward evaluation if every fundamental is pinned, or a single `scipy.optimize.least_squares`
call if you've given exact derived-quantity targets (like numax/dnu) instead. Either way this is
near-instant -- useful for tables of point estimates, or a quick sanity check before committing to a
full run.

In [ ]:
import time

# every fundamental pinned -> direct forward evaluation
t0 = time.time()
out = solver.solve(
    {"M": 1.0, "R": 1.0, "Teff": 5772.0, "plx": 10.0, "A_G": 0.0, "FeH": 0.0},
    want=["numax", "dnu", "G_mag"],
)
print(out, f"({time.time()-t0:.4f}s)")

# exact derived targets instead -> least-squares solves for M, R
t0 = time.time()
out = solver.solve(
    {"Teff": 5772.0, "FeH": 0.0, "plx": 10.0, "A_G": 0.0, "numax": 3090.0, "dnu": 135.1},
    want=["M", "R"],
)
print(out, f"({time.time()-t0:.4f}s)")

Mutually inconsistent exact constraints get flagged rather than silently returning a bad fit --
here dnu=500 muHz is nowhere near consistent with numax=3090 muHz at solar Teff, so the least-squares
solve can't satisfy both and warns about it.

In [ ]:
out = solver.solve(
    {"Teff": 5777.0, "FeH": 0.0, "plx": 10.0, "A_G": 0.0, "numax": 3090.0, "dnu": 500.0},
    want=["M", "R"],
)
print(out)

## 9. Custom priors

Override any fundamental parameter's prior: a `(mean, error)` tuple is shorthand for a Gaussian
(same convention as `given`), or pass a distribution directly for something else -- one of
AsteroScale's distribution classes, or a frozen `scipy.stats` distribution.

**Watch out:** if you provide a custom distribution for M, R, Teff, or plx,
make sure it has bounded, strictly-positive support (e.g. `TruncatedNormal` with a positive `low`,
not a plain `Normal`) -- these parameters are physically positive-only, and relations like
`logg = log10(M/R^2)` aren't defined for non-positive inputs. A bounded distribution also avoids
wasting nested-sampling proposals in unphysical regions.

In [ ]:
from asteroscale.distributions import TruncatedNormal

solver_custom = ast.Solver(
    priors={
        "Teff": (5750, 200),         # tuple shorthand -> Gaussian(5750, 200)
        "M": TruncatedNormal(loc=1.0, scale=0.1, low=0.1, high=3),  # a good mass
                                      # estimate -- bounded below at 0.1, not a
                                      # plain Normal (see the note above)
    },
    seed=0,
)
out = solver_custom.solve({"numax": (3090, 30), "dnu": (135.1, 1.0), "FeH": (0.0, 0.05)}, want=["Teff", "M", "R"])
ast.summarize(out)

## 10. Retrieving more quantities after the fact with `predict()`

`solve()` only returns what you asked for in `want`, but the full forward pass (all derived
quantities -- luminosity, density, envelope FWHM, oscillation amplitude, Gaia photometry, ...) is
computed every time regardless. `predict()` re-reads it from the *last* `solve()` call without
re-running the sampler, for both the nested-sampling and point-estimate paths.

In [ ]:
solver.solve(
    {"Teff": (5777, 50), "numax": (3090, 30), "dnu": (135.1, 1.0), "FeH": (0.0, 0.05)},
    want=["M", "R"],
)
extra = solver.predict(["L", "rho", "logg", "A_env", "FWHM_env"])
ast.summarize(extra)

## 11. Input validation

Obvious mistakes are caught before anything expensive runs: unknown quantity names, malformed
`(mean, error)` pairs, non-positive errors, and physically impossible values (negative mass, radius,
Teff, parallax, or extinction) all raise immediately.

In [ ]:
try:
    solver.solve({"M": -1.0, "Teff": 5772.0, "R": 1.0}, want=["numax"])
except ValueError as e:
    print("caught:", e)

try:
    solver.solve({"Teff": (5777, 50)}, want=["not_a_real_quantity"])
except KeyError as e:
    print("caught:", e)

Values that are merely unusual (rather than impossible) warn instead of erroring -- might be
exactly what you intend, but it's also the classic wrong-units typo.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    solver.solve(
        {"Teff": 50000.0, "M": 1.0, "R": 1.0, "plx": 10.0, "A_G": 0.0, "FeH": 0.0},
        want=["numax"],
    )
    for warning in w:
        print(warning.category.__name__, "-", warning.message)

## 12. Solving many targets in parallel

Each target is a completely independent problem -- `solve_many` just parallelizes the loop across
processes (one process per target, up to `n_jobs` at a time), it doesn't let targets share information
with each other. A target that fails (bad input, non-convergence, etc.) gets `{"_error": ...}` instead
of aborting the whole batch.

In [ ]:
targets = {
    "star_A": {"Teff": (5777, 50), "FeH": (0.0, 0.05), "numax": (3090, 30), "dnu": (135.1, 1.0)},
    "star_B": {"Teff": (5000, 60), "FeH": (-0.2, 0.05), "numax": (150, 5), "dnu": (12.5, 0.3)},
    "star_C": {"Teff": (6200, 60), "FeH": (0.1, 0.05), "numax": (1800, 50), "dnu": (85.0, 1.5)},
}

results = ast.solve_many(targets, want=["M", "R"], show_progress=True)

for target_id, out in results.items():
    if "_error" in out:
        print(f"{target_id}: FAILED -- {out['_error']}")
    else:
        print(f"{target_id}: M = {out['M'].mean():.2f} +/- {out['M'].std():.2f}, "
              f"R = {out['R'].mean():.2f} +/- {out['R'].std():.2f}")

## 13. Returning every available quantity with `want="all"`

Use `"all"` when exploring the model or building a table and you do not yet know which
outputs you will need. It returns the six fundamental quantities followed by every registered
derived relation; private metadata such as Dynesty's results object is not included.

In [ ]:
solar_all = solver.solve(
    {"M": 1.0, "R": 1.0, "Teff": 5772.0, "plx": 10.0, "A_G": 0.0, "FeH": 0.0},
    want="all",
)
for name, value in solar_all.items():
    print(f"{name:12s} {value:.5g}")

## 14. Choosing a Dynesty preset and inspecting the raw result

`fast` is useful during development, `standard` is the default, and `precise` spends more
time reducing sampling noise. A preset is a starting point: individual settings and `dlogz`
can still be overridden. `return_results=True` exposes Dynesty diagnostics when convergence
or effective sample size needs closer inspection.

In [ ]:
for preset in ("fast", "standard", "precise"):
    print(preset, ast.Solver(preset=preset).settings)
    print()

fast_solver = ast.Solver(preset="fast", seed=1)
fast_out = fast_solver.solve(
    {"Teff": (5772, 50), "FeH": (0.0, 0.05), "numax": (3090, 30),
     "dnu": (135.1, 1.0), "plx": 200.0, "A_G": 0.0},
    want=["M", "R"],
    return_results=True,
)
print("nested-sampling iterations:", fast_out["_results"].niter)

## 15. Predicting the oscillation envelope and granulation background

AsteroScale also returns the Ball et al. (2018) envelope amplitude/FWHM and the two
Kallinger et al. (2014) granulation components. The latter are the RMS amplitude and two
characteristic frequencies needed to construct the super-Lorentzian background.

In [ ]:
background = solver.solve(
    {"M": 1.0, "R": 1.0, "Teff": 5772.0, "plx": 1000.0, "A_G": 0.0, "FeH": 0.0},
    want=["A_env", "FWHM_env", "A_gran", "b_gran_low", "b_gran_high"],
)
background

## 16. Real stars: comparison with literature values

The seismic and spectroscopic measurements below are used as inputs; the listed masses,
radii, and luminosities are retained as comparison values and are **not** passed to the solver.
This makes the table a useful end-to-end sanity check. Agreement should be approximate: these
simple scaling relations are not expected to reproduce detailed modelling, dynamical masses,
or interferometric radii at their quoted sub-percent precision. The first two stars are
rough agreement checks. ε Indi A is deliberately retained as a stress case: its global
scaling-relation mass is significantly lower than the interferometry-assisted literature mass,
so it demonstrates that a successful run is not automatically an accurate result.

Sources:

- **α Cen A:** seismic inputs from Kjeldsen et al. (2005); dynamical mass, interferometric
  radius, luminosity, and Teff compiled by [Kervella et al. (2017)](https://doi.org/10.1051/0004-6361/201629505).
- **ε Tau:** seismic inputs and mass from [Arentoft et al. (2019)](https://doi.org/10.1051/0004-6361/201834690);
  interferometric radius and luminosity quoted in that analysis.
- **ε Indi A:** νmax and Δν from [Campante et al. (2024)](https://doi.org/10.1051/0004-6361/202449197);
  the comparison mass is from [Lundkvist et al. (2024)](https://doi.org/10.3847/1538-4357/ad25f2).

In [ ]:
import numpy as np

REAL_STARS = {
    "alpha Cen A": {
        "given": {"Teff": (5795, 19), "FeH": (0.23, 0.05),
                  "numax": (2410, 130), "dnu": (106.0, 0.5),
                  "plx": 747.17, "A_G": 0.0},
        "literature": {"M": (1.1055, 0.0039), "R": (1.2234, 0.0053),
                       "L": (1.521, 0.015)},
    },
    "epsilon Tau": {
        "given": {"Teff": (4950, 70), "FeH": (0.15, 0.05),
                  "numax": (56.4, 1.1), "dnu": (5.00, 0.01),
                  "plx": 22.37, "A_G": 0.0},
        "literature": {"M": (2.458, 0.073), "R": (12.46, 0.26),
                       "L": (79.4, 3.4)},
    },
    "epsilon Indi A": {
        "given": {"Teff": (4649, 7), "FeH": (-0.17, 0.05),
                  "numax": (5305, 176), "dnu": (201.25, 0.16),
                  "plx": 274.84, "A_G": 0.0},
        "literature": {"M": (0.782, 0.023)},
    },
}

def compare_with_literature(name, samples, literature):
    print(f"\n{name}")
    print(f"{'quantity':10s} {'AsteroScale p50 [p16, p84]':34s} literature")
    print("-" * 72)
    for quantity, (reference, error) in literature.items():
        p16, p50, p84 = np.percentile(samples[quantity], [16, 50, 84])
        print(
            f"{quantity:10s} {p50:8.3f} [{p16:8.3f}, {p84:8.3f}]"
            f"       {reference:8.3f} +/- {error:.3f}"
        )

literature_results = {}
for index, (name, data) in enumerate(REAL_STARS.items()):
    quantities = list(data["literature"])
    star_solver = ast.Solver(preset="fast", seed=100 + index)
    samples = star_solver.solve(data["given"], want=quantities)
    literature_results[name] = samples
    compare_with_literature(name, samples, data["literature"])